# Complete PCB SAM Extraction Pipeline (Dual GPU / T4 / P100)

Unified end-to-end pipeline that processes **both** the Kaggle (FICS) and WACV 2019 PCB datasets using GPU-accelerated SAM (Segment Anything Model).

### Kaggle Settings:
1. **Accelerator**: GPU T4 x2 (or P100)
2. **Internet**: ON
3. Click **Run All** -> leave for ~20-30 minutes -> download the output ZIP.

In [ ]:
# 1. Environment Setup & Dependency Installation
!pip install -q opencv-python numpy pandas openpyxl matplotlib kagglehub
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os
import sys
import json
import shutil
import urllib.request
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import List, Dict, Tuple, Any

import cv2
import numpy as np
import pandas as pd
import torch
from segment_anything import sam_model_registry, SamPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Compute Device:', device)
if device == 'cuda':
    print('GPU Count:', torch.cuda.device_count())
    print('GPU Model:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Download Pretrained SAM ViT-B Weights
sam_checkpoint = Path('sam_vit_b.pth')
if not sam_checkpoint.exists():
    print('Downloading sam_vit_b.pth (375 MB)...')
    urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth', str(sam_checkpoint))
    print('SAM checkpoint downloaded.')

print('Loading SAM on GPU...')
sam = sam_model_registry['vit_b'](checkpoint=str(sam_checkpoint))
sam.to(device=device)
sam.eval()
predictor = SamPredictor(sam)
print('SAM Predictor Ready.')

In [ ]:
# 3. Core Functions & Douglas-Peucker Polygon Extractor
def clean_box(x1: float, y1: float, x2: float, y2: float, img_w: int, img_h: int, min_size: int = 8) -> Tuple[bool, List[int]]:
    if x1 > x2: x1, x2 = x2, x1
    if y1 > y2: y1, y2 = y2, y1
    x1 = max(0, min(int(round(x1)), img_w - 1))
    y1 = max(0, min(int(round(y1)), img_h - 1))
    x2 = max(0, min(int(round(x2)), img_w))
    y2 = max(0, min(int(round(y2)), img_h))
    if (x2 - x1) < min_size or (y2 - y1) < min_size:
        return False, []
    return True, [x1, y1, x2, y2]

def extract_polygon(mask: np.ndarray, min_area: float = 15.0) -> List[List[float]]:
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return []
    cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(cnt) < min_area: return []
    approx = cv2.approxPolyDP(cnt, 0.005 * cv2.arcLength(cnt, True), True)
    return [[round(float(p[0][0]), 2), round(float(p[0][1]), 2)] for p in approx]

In [ ]:
# 4. PART 1: Process Kaggle / FICS Dataset
print('=' * 75)
print('STARTING PART 1: KAGGLE / FICS DATASET')
print('=' * 75)

import kagglehub
out_kaggle_dir = Path('outputs/kaggle_results')
out_kaggle_json = out_kaggle_dir / 'labelme_json'
out_kaggle_json.mkdir(parents=True, exist_ok=True)

# Locate or download dataset
possible_dirs = [Path('/kaggle/input/fics-pcb'), Path('/kaggle/input/pcb-component-detection'), Path('dataset_split/train/images')]
kaggle_img_dir = None
kaggle_lbl_dir = None

for p in possible_dirs:
    if p.exists():
        jpgs = list(p.rglob('*.jpg'))
        if jpgs:
            kaggle_img_dir = jpgs[0].parent
            candidate_lbl = kaggle_img_dir.parent / 'labels'
            if candidate_lbl.exists(): kaggle_lbl_dir = candidate_lbl
            break

if kaggle_img_dir is None:
    print('Downloading fics-pcb from KaggleHub...')
    hub_path = Path(kagglehub.dataset_download('ficslab/fics-pcb'))
    jpgs = list(hub_path.rglob('*.jpg'))
    if jpgs:
        kaggle_img_dir = jpgs[0].parent
        kaggle_lbl_dir = kaggle_img_dir.parent / 'labels'

print('Kaggle Images Directory:', kaggle_img_dir)
print('Kaggle Labels Directory:', kaggle_lbl_dir)

K_CLASSES = {0: 'Cap1', 1: 'Cap2', 2: 'Cap3', 3: 'Cap4', 4: 'MOSFET', 5: 'Mov', 6: 'Resistor', 7: 'Transformer'}
K_PREFIX = {'Cap1': 'C', 'Cap2': 'C', 'Cap3': 'C', 'Cap4': 'C', 'MOSFET': 'Q', 'Mov': 'D', 'Resistor': 'R', 'Transformer': 'T'}
K_FOOTPRINTS = {
    'Resistor': 'Resistor_SMD:R_0805_2012Metric', 'Cap1': 'Capacitor_SMD:C_0805_2012Metric',
    'Cap2': 'Capacitor_SMD:C_1206_3216Metric', 'Cap3': 'Capacitor_THT:CP_Radial_D6.3mm_P2.50mm',
    'Cap4': 'Capacitor_THT:CP_Radial_D8.0mm_P3.50mm', 'MOSFET': 'Package_TO_SOT_SMD:SOT-23',
    'Mov': 'Diode_SMD:D_SOD-123', 'Transformer': 'Transformer_SMD:Transformer_Bourns_SRF0703'
}

kaggle_images = sorted(list(kaggle_img_dir.glob('*.jpg')))
print(f'Total Kaggle Images to Process: {len(kaggle_images)}')

k_records = []
for idx, img_p in enumerate(kaggle_images, 1):
    img = cv2.imread(str(img_p))
    if img is None: continue
    h, w = img.shape[:2]
    lbl_p = kaggle_lbl_dir / f'{img_p.stem}.txt' if kaggle_lbl_dir else None
    if not lbl_p or not lbl_p.exists(): continue
    
    boxes = []
    with open(lbl_p, 'r') as f:
        for line in f:
            p = line.strip().split()
            if len(p) >= 5:
                cid = int(float(p[0]))
                xc, yc, bw, bh = map(float, p[1:5])
                ok, box = clean_box((xc - bw/2)*w, (yc - bh/2)*h, (xc + bw/2)*w, (yc + bh/2)*h, w, h)
                if ok: boxes.append((box, cid))
    if not boxes: continue
    
    predictor.set_image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    shapes = []
    ref_counts = {}
    for inst_id, (box, cid) in enumerate(boxes, 1):
        masks, scores, _ = predictor.predict(box=np.array(box)[None, :], multimask_output=False)
        pts = extract_polygon(masks[0])
        if not pts: pts = [[float(box[0]), float(box[1])], [float(box[2]), float(box[1])], [float(box[2]), float(box[3])], [float(box[0]), float(box[3])]]
        cname = K_CLASSES.get(cid, f'Class_{cid}')
        pfx = K_PREFIX.get(cname, 'U')
        ref_counts[pfx] = ref_counts.get(pfx, 0) + 1
        ref_des = f'{pfx}{ref_counts[pfx]}'
        k_records.append({
            'image': img_p.name, 'instance_id': inst_id, 'ref_des': ref_des, 'class': cname,
            'footprint': K_FOOTPRINTS.get(cname, ''), 'confidence': round(float(scores[0]), 3),
            'num_points': len(pts), 'points_compact': '; '.join([f'({pt[0]},{pt[1]})' for pt in pts]),
            'points_json': json.dumps(pts)
        })
        shapes.append({'label': f'{ref_des}: {cname}', 'points': pts, 'shape_type': 'polygon'})
    
    with open(out_kaggle_json / f'{img_p.stem}.json', 'w') as f:
        json.dump({'version': '5.5.0', 'flags': {}, 'shapes': shapes, 'imagePath': img_p.name, 'imageData': None, 'imageHeight': h, 'imageWidth': w}, f, indent=2)
    
    if idx % 100 == 0 or idx == len(kaggle_images):
        pd.DataFrame(k_records).to_csv(out_kaggle_dir / 'kaggle_sam_points.csv', index=False)
        print(f'[{idx}/{len(kaggle_images)}] Processed {len(k_records)} components on GPU...')

pd.DataFrame(k_records).to_excel(out_kaggle_dir / 'kaggle_sam_points.xlsx', index=False)
pd.DataFrame(k_records).to_csv(out_kaggle_dir / 'kaggle_sam_points.csv', index=False)
print('Kaggle processing finished! Total components:', len(k_records))

In [ ]:
# 5. PART 2: Process WACV 2019 Dataset (All 47 Boards)
print('=' * 75)
print('STARTING PART 2: WACV 2019 DATASET (47 BOARDS)')
print('=' * 75)

out_wacv_dir = Path('outputs/wacv_results')
out_wacv_dir.mkdir(parents=True, exist_ok=True)
wacv_root = Path('wacv_data/pcb_wacv_2019')

# Download WACV zip if not present
if not wacv_root.exists():
    wacv_zip = Path('wacv_data/pcb_wacv_2019.zip')
    wacv_zip.parent.mkdir(parents=True, exist_ok=True)
    if not wacv_zip.exists():
        print('Downloading WACV dataset from Georgia Tech RIPL (284 MB)...')
        urllib.request.urlretrieve('https://ripl.cc.gatech.edu/data/pcb_wacv_2019.zip', str(wacv_zip))
    print('Extracting WACV dataset...')
    with zipfile.ZipFile(wacv_zip, 'r') as zf:
        zf.extractall('wacv_data')
    print('WACV ready.')

W_CLASSES = {
    'capacitor': {'kicad': 'Capacitor_SMD', 'footprint': 'Capacitor_SMD:C_0805_2012Metric', 'ref_prefix': 'C'},
    'electrolytic': {'kicad': 'Capacitor_THT', 'footprint': 'Capacitor_THT:CP_Radial_D6.3mm_P2.50mm', 'ref_prefix': 'C'},
    'resistor': {'kicad': 'Resistor_SMD', 'footprint': 'Resistor_SMD:R_0805_2012Metric', 'ref_prefix': 'R'},
    'ic': {'kicad': 'Package_SO', 'footprint': 'Package_SO:SOIC-8_3.9x4.9mm_P1.27mm', 'ref_prefix': 'U'},
    'transistor': {'kicad': 'Package_TO_SOT_SMD', 'footprint': 'Package_TO_SOT_SMD:SOT-23', 'ref_prefix': 'Q'},
    'diode': {'kicad': 'Diode_SMD', 'footprint': 'Diode_SMD:D_SOD-123', 'ref_prefix': 'D'},
    'connector': {'kicad': 'Connector', 'footprint': 'Connector_PinHeader_2.54mm:PinHeader_1x04_P2.54mm_Vert', 'ref_prefix': 'J'},
    'inductor': {'kicad': 'Inductor_SMD', 'footprint': 'Inductor_SMD:L_0805_2012Metric', 'ref_prefix': 'L'},
    'switch': {'kicad': 'Button_Switch_SMD', 'footprint': 'Button_Switch_SMD:SW_Push_SPST_NO_Alps_SKRK', 'ref_prefix': 'SW'},
    'button': {'kicad': 'Button_Switch_SMD', 'footprint': 'Button_Switch_SMD:SW_Push_SPST_NO_Alps_SKRK', 'ref_prefix': 'SW'},
    'led': {'kicad': 'LED_SMD', 'footprint': 'LED_SMD:LED_0805_2012Metric', 'ref_prefix': 'D'},
    'clock': {'kicad': 'Crystal', 'footprint': 'Crystal:Crystal_SMD_3225-4Pin_3.2x2.5mm', 'ref_prefix': 'Y'},
    'fuse': {'kicad': 'Fuse', 'footprint': 'Fuse:Fuse_1206_3216Metric', 'ref_prefix': 'F'},
    'transformer': {'kicad': 'Transformer_SMD', 'footprint': 'Transformer_SMD:Transformer_Bourns_SRF0703', 'ref_prefix': 'T'}
}
IGNORED_WACV = {'text', 'pads', 'pins', 'unknown', 'test'}

board_folders = sorted([d for d in wacv_root.iterdir() if d.is_dir() and list(d.glob('*.xml'))])
print(f'Total WACV Boards to Process: {len(board_folders)}')

w_records = []
for b_idx, b_dir in enumerate(board_folders, 1):
    xml_p = list(b_dir.glob('*.xml'))[0]
    imgs = list(b_dir.glob('*.jpg')) + list(b_dir.glob('*.png'))
    if not imgs: continue
    img_p = [p for p in imgs if p.suffix.lower() == '.jpg']
    img_p = img_p[0] if img_p else imgs[0]
    
    img = cv2.imread(str(img_p))
    if img is None: continue
    h, w = img.shape[:2]
    
    tree = ET.parse(xml_p)
    boxes = []
    for obj in tree.getroot().findall('object'):
        raw_name = obj.find('name').text if obj.find('name') is not None else ''
        first_word = raw_name.strip().strip('"').lower().split()[0] if raw_name else ''
        if first_word in IGNORED_WACV: continue
        bnd = obj.find('bndbox')
        if bnd is None: continue
        ok, box = clean_box(float(bnd.find('xmin').text), float(bnd.find('ymin').text), float(bnd.find('xmax').text), float(bnd.find('ymax').text), w, h)
        if ok: boxes.append((box, first_word, raw_name))
    
    if not boxes: continue
    predictor.set_image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    shapes = []
    ref_counts = {}
    for inst_id, (box, first_word, raw_name) in enumerate(boxes, 1):
        masks, scores, _ = predictor.predict(box=np.array(box)[None, :], multimask_output=False)
        pts = extract_polygon(masks[0])
        if not pts: pts = [[float(box[0]), float(box[1])], [float(box[2]), float(box[1])], [float(box[2]), float(box[3])], [float(box[0]), float(box[3])]]
        info = W_CLASSES.get(first_word, {'kicad': 'Generic_Component', 'footprint': 'Package_SO:SOIC-8', 'ref_prefix': 'U'})
        pfx = info['ref_prefix']
        ref_counts[pfx] = ref_counts.get(pfx, 0) + 1
        ref_des = f'{pfx}{ref_counts[pfx]}'
        w_records.append({
            'board': b_dir.name, 'ref_des': ref_des, 'type': first_word, 'kicad_footprint': info['footprint'],
            'confidence': round(float(scores[0]), 3), 'num_points': len(pts),
            'points_compact': '; '.join([f'({pt[0]},{pt[1]})' for pt in pts]), 'points_json': json.dumps(pts)
        })
        shapes.append({'label': f'{ref_des}: {first_word.upper()}', 'points': pts, 'shape_type': 'polygon'})
    
    # Save LabelMe JSON & copy image
    shutil.copy2(img_p, out_wacv_dir / img_p.name)
    with open(out_wacv_dir / f'{img_p.stem}.json', 'w') as f:
        json.dump({'version': '5.5.0', 'flags': {}, 'shapes': shapes, 'imagePath': img_p.name, 'imageData': None, 'imageHeight': h, 'imageWidth': w}, f, indent=2)
    print(f'[{b_idx}/{len(board_folders)}] Board {b_dir.name}: {len(boxes)} components segmented.')

pd.DataFrame(w_records).to_excel(out_wacv_dir / 'wacv_all_boards_sam_points.xlsx', index=False)
pd.DataFrame(w_records).to_csv(out_wacv_dir / 'wacv_all_boards_sam_points.csv', index=False)
print('WACV processing complete! Total components:', len(w_records))

In [ ]:
# 6. Package All Outputs into 1-Click Downloadable ZIP
print('Zipping all Kaggle and WACV results...')
shutil.make_archive('pcb_sam_complete_results', 'zip', 'outputs')
print('SUCCESS: Created pcb_sam_complete_results.zip in /kaggle/working/')
print('You can download this zip directly from the Kaggle Output sidebar!')